In [1]:
import os
from src.arguments import ModelArguments, DataArguments
from src.model.model import MMEBModel
from src.model.processor import load_processor, QWEN2_VL, VLM_IMAGE_TOKENS, LLAVA_ONEVISION, \
    Qwen2_VL_process_fn, LLAVA_QWEN2, FastVLM_process_fn, Llava_ONEVISION_process_fn
from src.utils import batch_to_device
from PIL import Image
import numpy as np
from src.model.llava.model import LlavaQwen2ForCausalLM
import torch
import math
%matplotlib inline
import matplotlib.pyplot as plt
import torch.nn.functional as F

from transformers.image_transforms import (
    convert_to_rgb,
    resize,
)

/home/hungpv/projects/Talas_VLM_Embed/vlm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/hungpv/projects/Talas_VLM_Embed/src/model/vlm_backbone/internvideo2/modeling_internvideo2.py:541: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
[2026-09-09 05:44:29,469] DEBUG [matplotlib:347] matplotlib data path: /home/hungpv/projects/Talas_VLM_Embed/vlm/lib/python3.12/site-packages/matplotlib/mpl-data
[2026-09-09 05:44:29,477] DEBUG [matplotlib:347] CONFIGDIR=/mnt/hungpv/.config/matplotlib
[2026-09-09 05:44:29,493] DEBUG [matplotlib:1564] interactive is False
[2026-09-09 05:44:29,494] DEBUG [matplotlib:1565] platform is linux
[2026-09-09 05:44:29,528] DEBUG [matplotlib:347] CACHEDIR=/mnt/hu

FusedMLP of flash_attn is not installed!!!
DropoutAddRMSNorm of flash_attn is not installed!!!
flash_attn_interface or bert_padding of flash_attn is not installed!!!


[2026-09-09 05:44:29,531] DEBUG [matplotlib.font_manager:1843] Using fontManager instance from /mnt/hungpv/.cache/matplotlib/fontlist-v3.11.0.json
[2026-09-09 05:44:29,879] DEBUG [matplotlib.pyplot:517] Loaded backend module://matplotlib_inline.backend_inline version unknown.
[2026-09-09 05:44:29,883] DEBUG [matplotlib.pyplot:517] Loaded backend inline version unknown.


In [2]:
# model_args = ModelArguments(
#     model_name='raghavlite/B3_Qwen2_2B',
#     pooling='last',
#     normalize=True,
#     model_backbone='qwen2_vl',
#     lora=True
# )
# process_fn = Qwen2_VL_process_fn
# token_img = QWEN2_VL

# model_args = ModelArguments(
#     model_name='apple/FastVLM-0.5B',
#     pooling='last',
#     normalize=True,
#     model_backbone=LLAVA_QWEN2,
#     lora=True,
# )
# process_fn = FastVLM_process_fn
# token_img = LLAVA_QWEN2

model_args = ModelArguments(
    model_name='llava-hf/llava-onevision-qwen2-0.5b-ov-hf',
    pooling='last',
    normalize=True,
    model_backbone=LLAVA_ONEVISION,
    lora=True,
)
process_fn = Llava_ONEVISION_process_fn
token_img = LLAVA_ONEVISION

data_args = DataArguments(
    image_resolution = 'tiny'
)

processor = load_processor(model_args, None)
model = MMEBModel.build(model_args)
model = model.to('cuda:1', dtype=torch.bfloat16)
model.eval()

[2026-09-09 05:44:29,895] INFO [src.utils:21] Loading processor from: llava-hf/llava-onevision-qwen2-0.5b-ov-hf
[2026-09-09 05:44:29,898] DEBUG [urllib3.connectionpool:1062] Starting new HTTPS connection (1): huggingface.co:443
[2026-09-09 05:44:30,167] DEBUG [urllib3.connectionpool:544] https://huggingface.co:443 "HEAD /llava-hf/llava-onevision-qwen2-0.5b-ov-hf/resolve/main/tokenizer_config.json HTTP/1.1" 307 0
[2026-09-09 05:44:30,176] DEBUG [urllib3.connectionpool:544] https://huggingface.co:443 "HEAD /api/resolve-cache/models/llava-hf/llava-onevision-qwen2-0.5b-ov-hf/74dd0bf867a4cda7950c17663794267c60cf4b40/tokenizer_config.json HTTP/1.1" 200 0
[2026-09-09 05:44:30,412] DEBUG [urllib3.connectionpool:544] https://huggingface.co:443 "GET /api/models/llava-hf/llava-onevision-qwen2-0.5b-ov-hf/tree/main/additional_chat_templates?recursive=False&expand=False HTTP/1.1" 404 64
[2026-09-09 05:44:30,905] DEBUG [urllib3.connectionpool:544] https://huggingface.co:443 "GET /api/models/llava-hf/

Detected model type: llava_onevision
Determined model backbone: llava_onevision


[2026-09-09 05:44:39,323] DEBUG [urllib3.connectionpool:544] https://huggingface.co:443 "HEAD /llava-hf/llava-onevision-qwen2-0.5b-ov-hf/resolve/main/generation_config.json HTTP/1.1" 307 0
[2026-09-09 05:44:39,332] DEBUG [urllib3.connectionpool:544] https://huggingface.co:443 "HEAD /api/resolve-cache/models/llava-hf/llava-onevision-qwen2-0.5b-ov-hf/74dd0bf867a4cda7950c17663794267c60cf4b40/generation_config.json HTTP/1.1" 200 0
[2026-09-09 05:44:39,558] DEBUG [urllib3.connectionpool:544] https://huggingface.co:443 "HEAD /llava-hf/llava-onevision-qwen2-0.5b-ov-hf/resolve/main/custom_generate/generate.py HTTP/1.1" 404 0
[2026-09-09 05:44:39,562] INFO [src.utils:21] Initializing LoRA adapter from LlavaOnevisionForConditionalGeneration(
  (model): LlavaOnevisionModel(
    (vision_tower): SiglipVisionModel(
      (vision_model): SiglipVisionTransformer(
        (embeddings): SiglipVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=va

Found 120 explicit target modules
First 5 modules: ['model.language_model.layers.0.self_attn.q_proj', 'model.language_model.layers.0.self_attn.k_proj', 'model.language_model.layers.0.self_attn.v_proj', 'model.language_model.layers.0.self_attn.o_proj', 'model.language_model.layers.0.mlp.down_proj']
Applying LoRA to vision_tower/layers: ['qkv_proj', 'o_proj', 'gate_up_proj', 'down_proj', 'k_proj', 'q_proj', 'out_proj', 'v_proj']


MMEBModel(
  (encoder): PeftModel(
    (base_model): LoraModel(
      (model): LlavaOnevisionForConditionalGeneration(
        (model): LlavaOnevisionModel(
          (vision_tower): SiglipVisionModel(
            (vision_model): SiglipVisionTransformer(
              (embeddings): SiglipVisionEmbeddings(
                (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
                (position_embedding): Embedding(729, 1152)
              )
              (encoder): SiglipEncoder(
                (layers): ModuleList(
                  (0-25): 26 x SiglipEncoderLayer(
                    (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
                    (self_attn): SiglipAttention(
                      (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
                      (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
                      (q_proj): Linear(in_features=1152, out_features=

In [3]:
processor_inputs = {
    "text": [f'Represent the given image with the following question: What is in the image {VLM_IMAGE_TOKENS[token_img]}',
          f'{VLM_IMAGE_TOKENS[token_img]} Represent the given image with the following question: What is in the image'],
    "images": [Image.open('example.jpg').resize((500, 1025)),
            Image.open('example.jpg')],
}

inputs = process_fn(
    processor_inputs,
    processor, 
    # square_padding=True
    )
inputs = batch_to_device(inputs, "cuda")
# inputs['images'][0], inputs['images'][1]



[2026-09-09 05:44:48,672] DEBUG [PIL.Image:421] Importing JpegImagePlugin


In [4]:
input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]

eos_id = processor.tokenizer.eos_token_id
last_idx = attention_mask.long().sum(dim=1) - 1
last_ids = input_ids[torch.arange(input_ids.size(0)), last_idx]

print("eos_id:", eos_id)
print("last_ids:", last_ids.tolist())
print("all end with eos:", bool((last_ids == eos_id).all()))
print("all special ids: ", processor.tokenizer.all_special_ids)

eos_id: 151645
last_ids: [151645, 151643]
all end with eos: False
all special ids:  [151645, 151643, 151644]


In [8]:
print(input_ids.shape, attention_mask.shape)
print("input_ids:", (input_ids[1]==151646).sum())


torch.Size([2, 4066]) torch.Size([2, 4066])
input_ids: tensor(1737, device='cuda:0')


In [ ]:
type(model.encoder.get_vision_tower().vision_tower.model)

In [ ]:
output = model.encode_input(inputs)
output

In [ ]:
x = torch.randn(1, 3, 768, 768).to('cuda', dtype=torch.bfloat16)
with torch.no_grad():
    y = model.encoder.get_vision_tower()(x)
y.shape